# HPRC Two-Annotation Overview — Agreement and Method-Specific Content

This notebook summarizes where Ensembl and CAT annotations agree and where they differ across HPRC assemblies, reusing loading helpers and palettes from the main concordance analysis. Figures save to `$HPRC_QC_OUTPUT_DIR/figures` (or the default cluster path if unset).

In [ ]:

import os, shutil
from pathlib import Path
import matplotlib as mpl
from matplotlib import font_manager as fm

# Use local scratch for Matplotlib cache to avoid stale NFS handles
try:
    MPLDIR = Path('/tmp') / f"{os.environ.get('USER','user')}-mplconfig"
    MPLDIR.mkdir(parents=True, exist_ok=True)
    os.environ['MPLCONFIGDIR'] = str(MPLDIR)
    import matplotlib
    ttf_src = Path(matplotlib.get_data_path())/'fonts'/'ttf'
    ttf_dst = MPLDIR/'ttf'
    shutil.copytree(ttf_src, ttf_dst, dirs_exist_ok=True)
    for f in ttf_dst.glob('*.ttf'):
        fm.fontManager.addfont(str(f))
    fm._load_fontmanager(try_read_cache=False)
    mpl.rcParams.update({'svg.fonttype':'none', 'pdf.fonttype':42, 'ps.fonttype':42,
                         'font.family':'DejaVu Sans'})
except Exception as e:
    print('Font init warning:', e)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Publication-quality defaults (match v2)
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 8,
    'axes.labelsize': 9,
    'axes.titlesize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 7,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
})

print(f'pandas {pd.__version__}, numpy {np.__version__}')


In [ ]:

# Paths
DEFAULT_OUT = '/hps/nobackup/flicek/ensembl/genebuild/jackt/hprc/hprc-qc/results'
OUTPUT_DIR  = Path(os.getenv('HPRC_QC_OUTPUT_DIR', DEFAULT_OUT))
QC_DIR         = OUTPUT_DIR / 'qc_metrics'
RESULTS_DIR    = OUTPUT_DIR / 'results'
SUMMARY_DIR    = OUTPUT_DIR / 'summary_stats'
FIGURE_DIR     = OUTPUT_DIR / 'figures'
INTRON_DIR     = OUTPUT_DIR / 'intermediate_spreadsheets' / 'intron_chain'
CDS_DIR        = OUTPUT_DIR / 'intermediate_spreadsheets' / 'coding_integrity'
DIV_DIR        = OUTPUT_DIR / 'intermediate_spreadsheets' / 'divergence'
FIGURE_DIR.mkdir(exist_ok=True, parents=True)


def save_panel(fig, prefix):
    """Save a standalone panel as PDF + PNG + SVG."""
    for ext in ('pdf', 'png', 'svg'):
        kw = {'dpi': 300} if ext == 'png' else {}
        fig.savefig(FIGURE_DIR / f'{prefix}.{ext}', bbox_inches='tight', **kw)
    print(f'  Saved {prefix}.{{pdf,png,svg}} → {FIGURE_DIR}')


def save_data(df, prefix):
    path = FIGURE_DIR / f'{prefix}_data.tsv'
    df.to_csv(path, sep='	', index=False)
    print(f'  Saved {prefix}_data.tsv  ({len(df)} rows)')

print(f'Output:  {OUTPUT_DIR}')
print(f'Figures: {FIGURE_DIR}')


In [ ]:

# Colours and labels
CLASS_COLORS = {
    'Exact_Match':    '#2a9d8f',
    'Intron_Match':   '#264653',
    'Intron_Subset':  '#457b9d',
    'Intron_Superset':'#74a9cf',
    'Partial_5':      '#e9c46a',
    'Partial_3':      '#f4a261',
    'Other_Partial':  '#e76f51',
    'No_Match':       '#c1121f',
}
CLASS_LABELS = {
    'Exact_Match':     'Exact match',
    'Intron_Match':    'Same intron chain',
    'Intron_Subset':   'Same intron chain',
    'Intron_Superset': 'Same intron chain',
    'Partial_5':       'Partial overlap',
    'Partial_3':       'Partial overlap',
    'Other_Partial':   'Partial overlap',
    'No_Match':        'No match',
}
CLASSIFICATION_ORDER = [
    'Exact_Match', 'Intron_Match', 'Intron_Subset', 'Intron_Superset',
    'Partial_5', 'Partial_3', 'Other_Partial', 'No_Match',
]

GROUP_4_MAP = {
    'Exact_Match':    'Exact match',
    'Intron_Match':   'Same intron chain',
    'Intron_Subset':  'Same intron chain',
    'Intron_Superset':'Same intron chain',
    'Partial_5':      'Partial overlap',
    'Partial_3':      'Partial overlap',
    'Other_Partial':  'Partial overlap',
    'No_Match':       'No match',
}
GROUP_4_COLORS = {
    'Exact match':       '#2a9d8f',
    'Same intron chain': '#457b9d',
    'Partial overlap':   '#f4a261',
    'No match':          '#c1121f',
}
GROUP_4_ORDER = ['Exact match', 'Same intron chain', 'Partial overlap', 'No match']

BIOTYPE_ORDER  = ['protein_coding', 'lncRNA', 'pseudogene', 'other_ncRNA', 'other']
BIOTYPE_LABELS = {
    'protein_coding': 'Protein-coding',
    'lncRNA':         'lncRNA',
    'pseudogene':     'Pseudogene',
    'other_ncRNA':    'Other ncRNA',
    'other':          'Other',
}


In [ ]:

# Gene presence per assembly
funnel_file = SUMMARY_DIR / 'funnel_rung1_gene_presence_per_asm.tsv'
if funnel_file.exists():
    gene_pres = pd.read_csv(funnel_file, sep='	')
    print(f'Loaded gene presence: {len(gene_pres)} assemblies')
else:
    import re
    ACC_RE = re.compile(r'(GC[AF]_\d+\.\d+)')
    files = sorted(QC_DIR.rglob('*_gene_presence.tsv'))
    print(f'Computing gene presence from {len(files)} files...')
    rows = []
    for f in files:
        m = ACC_RE.search(f.name)
        acc = m.group(1) if m else f.stem
        df = pd.read_csv(f, sep='	')
        for col in ['present_in_ensembl', 'present_in_cat']:
            df[col] = df[col].map({'True': True, 'False': False, True: True, False: False})
        df = df[~df['gene_name'].str.match(r'^ENSG', na=False)]
        n_union = len(df)
        n_both = ((df['present_in_ensembl']) & (df['present_in_cat'])).sum()
        rows.append({'assembly_accession': acc,
                     'n_union_loci': n_union, 'n_both_loci': int(n_both),
                     'pct_gene_presence': n_both / n_union if n_union > 0 else np.nan})
    gene_pres = pd.DataFrame(rows)
    print(f'Computed gene presence: {len(gene_pres)} assemblies')

gene_pres['pct'] = gene_pres['pct_gene_presence'] * 100

# Intron chain classification
ic_file = INTRON_DIR / 'intron_chain_by_biotype_per_assembly.tsv'
if ic_file.exists():
    ic_data = pd.read_csv(ic_file, sep='	')
    print(f'Loaded intron chain: {len(ic_data):,} rows, '
          f"{ic_data['assembly_accession'].nunique()} assemblies")
    if 'direction' not in ic_data.columns:
        ic_data['direction'] = 'Ensembl_to_CAT'
        print('  (legacy format — added direction=Ensembl_to_CAT)')
else:
    print(f'WARNING: {ic_file} not found — run aggregate_intron_chain_by_biotype.py first')
    ic_data = pd.DataFrame()

# Jaccard by biotype
jac_file = INTRON_DIR / 'jaccard_by_biotype_per_assembly.tsv'
jac_data = pd.read_csv(jac_file, sep='	') if jac_file.exists() else pd.DataFrame()
if not jac_data.empty:
    print(f'Loaded Jaccard: {len(jac_data):,} rows')

# CDS concordance
cds_asm_file = CDS_DIR / 'coding_integrity_per_assembly.tsv'
cds_asm = pd.read_csv(cds_asm_file, sep='	') if cds_asm_file.exists() else pd.DataFrame()
if not cds_asm.empty:
    print(f'Loaded CDS: {len(cds_asm)} assemblies')

# Divergence summary (for method-specific content)
div_asm_file = DIV_DIR / 'grch38_divergence_per_assembly.tsv'
div_asm = pd.read_csv(div_asm_file, sep='	') if div_asm_file.exists() else pd.DataFrame()
if not div_asm.empty:
    print(f'Loaded divergence: {div_asm["assembly_accession"].nunique()} assemblies')

# Feature breakdown counts

def group_biotype_simple(b: str) -> str:
    b = str(b or '').lower()
    if 'protein_coding' in b:
        return 'protein_coding'
    if 'lncrna' in b or 'lnc_rna' in b:
        return 'lncRNA'
    if 'pseudogene' in b or 'pseudogenic' in b:
        return 'pseudogene'
    if any(x in b for x in ['snrna','snorna','mirna','trna','rrna','ncrna','antisense','tec','guide_rna','scrna','vault_rna','y_rna']):
        return 'other_ncRNA'
    return 'other'

ensembl_files = sorted(QC_DIR.rglob('*_gene_transcript_counts.tsv'))
ensembl_files = [p for p in ensembl_files if '_cat_' not in p.name]
cat_files = sorted(QC_DIR.rglob('*_cat_gene_transcript_counts.tsv'))

ens_list, cat_list = [], []
usecols = ['assembly_accession','gene_id','biotype','n_transcripts']
for p in ensembl_files:
    try:
        df = pd.read_csv(p, sep='	', usecols=usecols, dtype={'gene_id':str,'biotype':str})
        df['source'] = 'Ensembl'; ens_list.append(df)
    except Exception as e:
        print('WARN (Ensembl counts):', p, e)
for p in cat_files:
    try:
        df = pd.read_csv(p, sep='	', usecols=usecols, dtype={'gene_id':str,'biotype':str})
        df['source'] = 'CAT'; cat_list.append(df)
    except Exception as e:
        print('WARN (CAT counts):', p, e)

counts_df = pd.concat(ens_list + cat_list, ignore_index=True) if (ens_list or cat_list) else pd.DataFrame()
if counts_df.empty:
    print('No gene transcript count tables found — feature breakdown panels will be skipped.')
else:
    counts_df['biotype_group'] = counts_df['biotype'].map(group_biotype_simple)
    counts_df['biotype_group'] = pd.Categorical(counts_df['biotype_group'], categories=BIOTYPE_ORDER, ordered=True)
    print('Counts loaded:', len(counts_df), 'rows', '| assemblies =', counts_df['assembly_accession'].nunique())


In [ ]:

# Compute median + IQR per direction × biotype × classification, and collapsed 4-group
medians_by_dir, stats_by_dir = {}, {}
medians_4_by_dir, stats_4_by_dir = {}, {}

if not ic_data.empty:
    for direction in ['Ensembl_to_CAT', 'CAT_to_Ensembl']:
        dir_data = ic_data[ic_data['direction'] == direction].copy()
        if dir_data.empty:
            continue
        stats = (
            dir_data.groupby(['biotype', 'classification'])['pct']
            .agg(
                median_pct='median',
                q25_pct=lambda x: x.quantile(0.25),
                q75_pct=lambda x: x.quantile(0.75),
                n_assemblies='count'
            )
            .reset_index()
        )
        stats_by_dir[direction] = stats

        medians = (
            stats.pivot(index='biotype', columns='classification', values='median_pct')
            .reindex(index=BIOTYPE_ORDER, columns=CLASSIFICATION_ORDER, fill_value=0)
        )
        medians_by_dir[direction] = medians

        dir_data['group'] = dir_data['classification'].map(GROUP_4_MAP)
        collapsed = (
            dir_data.groupby(['assembly_accession', 'biotype', 'group'], as_index=False)['pct']
            .sum()
        )
        stats_4 = (
            collapsed.groupby(['biotype', 'group'])['pct']
            .agg(
                median_pct='median',
                q25_pct=lambda x: x.quantile(0.25),
                q75_pct=lambda x: x.quantile(0.75),
                n_assemblies='count'
            )
            .reset_index()
        )
        stats_4_by_dir[direction] = stats_4
        medians_4 = (
            stats_4.pivot(index='biotype', columns='group', values='median_pct')
            .reindex(index=BIOTYPE_ORDER, columns=GROUP_4_ORDER, fill_value=0)
        )
        medians_4_by_dir[direction] = medians_4
else:
    print('No intron chain data — B panels will be skipped.')


In [ ]:

# ── A0: Method-specific content (median across assemblies) ───────────────
if div_asm.empty:
    print('Skipping A0 — need divergence summary (div_asm).')
else:
    cols = ['pct_both_agree_reference','pct_both_agree_diverged',
            'pct_ensembl_specific','pct_cat_specific']
    sub_cols = ['assembly_accession'] + [c for c in cols if c in div_asm.columns]
    sub = div_asm[sub_cols].copy().fillna(0)
    sub['pct_shared'] = sub.get('pct_both_agree_reference', 0) + sub.get('pct_both_agree_diverged', 0)

    med = sub[[c for c in ['pct_ensembl_specific','pct_cat_specific','pct_shared'] if c in sub.columns]].median()
    ens_med = float(med.get('pct_ensembl_specific', 0))
    cat_med = float(med.get('pct_cat_specific', 0))
    shared_med = float(med.get('pct_shared', 0))

    fig, ax = plt.subplots(figsize=(5.4, 2.8))
    ax.barh(['Method-specific'], [-ens_med], height=0.6, color='#e67e22', alpha=0.85, edgecolor='white')
    ax.barh(['Method-specific'], [ cat_med], height=0.6, color='#9b59b6', alpha=0.85, edgecolor='white')

    rng = np.random.default_rng(7)
    jitter = rng.uniform(-0.1, 0.1, len(sub))
    if 'pct_ensembl_specific' in sub.columns:
        ax.scatter(-sub['pct_ensembl_specific'], np.zeros(len(sub))+jitter, s=6, color='#e67e22', alpha=0.35, zorder=3)
    if 'pct_cat_specific' in sub.columns:
        ax.scatter( sub['pct_cat_specific'],     np.zeros(len(sub))+jitter, s=6, color='#9b59b6', alpha=0.35, zorder=3)

    ax.set_xlabel('Percent of genes per assembly')
    ax.set_yticks([])
    m = max(ens_med, cat_med, 1)
    ax.set_xlim(-m*1.6, m*1.6)
    ax.axvline(0, color='#999', linewidth=0.8)
    ax.grid(axis='x', color='#ddd', linewidth=0.5, alpha=0.7); ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    ax.text(-ens_med, 0.18, f'Ensembl-only {ens_med:.1f}%', ha='right', va='bottom', fontsize=8, fontweight='bold')
    ax.text( cat_med, 0.18, f'CAT-only {cat_med:.1f}%',   ha='left',  va='bottom', fontsize=8, fontweight='bold')

    plt.tight_layout()
    save_panel(fig, 'overview_A0_method_specific_butterfly')
    plt.show()


In [ ]:

# ── A: Shared gene loci coverage ─────────────────────────────────────────
vals = gene_pres['pct'].dropna()
if len(vals) == 0:
    print('Skipping A — no gene presence percentages available.')
else:
    med = vals.median()
    fig, ax = plt.subplots(figsize=(6.0, 2.6))
    ax.barh(0, med, height=0.5, color='#2a9d8f', alpha=0.85, edgecolor='white')
    ax.text(med + 0.2, 0, f'{med:.1f}%', va='center', fontsize=9, fontweight='bold')
    rng = np.random.default_rng(42)
    jitter_y = rng.uniform(-0.18, 0.18, len(vals))
    ax.scatter(vals, jitter_y, s=6, alpha=0.4, color='#264653', edgecolors='none', zorder=3)
    ax.set_xlabel('Gene loci detected by both methods (%)')
    ax.set_yticks([])
    ax.set_xlim(max(vals.min() - 1, 80), 101)
    ax.axvline(med, color='#264653', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.grid(axis='x', color='#ddd', linewidth=0.5, alpha=0.7); ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.tight_layout()
    save_panel(fig, 'overview_A_shared_coverage')
    plt.show()


In [ ]:

# ── B: Transcript concordance by biotype (collapsed 4-group; both directions) ───
if not ic_data.empty and 'Ensembl_to_CAT' in medians_4_by_dir:
    fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.0), sharey=True, constrained_layout=True)
    y_pos = np.arange(len(BIOTYPE_ORDER))
    for ax, direction, title in [
        (axes[0], 'Ensembl_to_CAT', 'Ensembl → CAT'),
        (axes[1], 'CAT_to_Ensembl', 'CAT → Ensembl'),
    ]:
        if direction not in medians_4_by_dir:
            ax.set_visible(False); continue
        med4 = medians_4_by_dir[direction]
        left = np.zeros(len(BIOTYPE_ORDER))
        for grp in GROUP_4_ORDER:
            v = med4[grp].values
            ax.barh(y_pos, v, left=left, height=0.65,
                    color=GROUP_4_COLORS[grp], edgecolor='white', linewidth=0.3)
            left += v
        ax.set_yticks(y_pos)
        if ax is axes[0]:
            ax.set_yticklabels([BIOTYPE_LABELS[b] for b in BIOTYPE_ORDER])
        else:
            ax.set_yticklabels([])
        ax.set_xlim(0, 100); ax.invert_yaxis()
        ax.set_title(title, fontsize=10, fontweight='bold', loc='left')
        ax.set_xlabel('Pct of transcripts (median across assemblies)')
        ax.grid(axis='x', color='#eee', linewidth=0.5, alpha=0.8); ax.set_axisbelow(True)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

        # Explicit agreement callout
        if set(['Exact match','Same intron chain']).issubset(med4.columns):
            good = float((med4['Exact match'] + med4['Same intron chain']).median())
            ax.text(99.5, 0.6, f'Exact+Same = {good:.0f}%', ha='right',
                    fontsize=9, fontweight='bold', color='#222')

    handles = [mpatches.Patch(color=GROUP_4_COLORS[g], label=g) for g in GROUP_4_ORDER]
    fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, -0.06), ncol=4, fontsize=7, frameon=False)
    save_panel(fig, 'overview_B_4group_directions'); plt.show()
else:
    print('Skipping B — missing intron chain medians.')


In [ ]:

# ── C: Jaccard exon overlap by biotype ───────────────────────────────────
if jac_data.empty:
    print('Skipping C — no Jaccard data')
else:
    fig, ax = plt.subplots(figsize=(7.0, 4.2))
    bio_stats, labels = [], []
    for b in BIOTYPE_ORDER:
        sub = jac_data[jac_data['biotype']==b]
        if sub.empty: continue
        bio_stats.append({
            'med': float(sub['median'].median()),
            'q1': float(sub['p25'].median()), 'q3': float(sub['p75'].median()),
            'whislo': float(sub['p5'].median()), 'whishi': float(sub['p95'].median()),
            'fliers': [],
        })
        labels.append(BIOTYPE_LABELS[b])
    pos = np.arange(1, len(bio_stats)+1)
    bp = ax.bxp(bio_stats, positions=pos, widths=0.55, patch_artist=True, showfliers=False, medianprops=dict(color='black', linewidth=2))
    for patch in bp['boxes']:
        patch.set_facecolor('#2a9d8f'); patch.set_alpha(0.6)
    ax.set_xticks(pos); ax.set_xticklabels(labels, rotation=20, ha='right')
    ax.set_ylabel('Jaccard index (best-match exon overlap)')
    ax.set_ylim(-0.05, 1.05); ax.axhline(1.0, color='grey', linewidth=0.5, linestyle=':', alpha=0.4)
    ax.grid(axis='y', color='#eee', linewidth=0.5, alpha=0.8); ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.tight_layout(); save_panel(fig, 'overview_C_jaccard_by_biotype'); plt.show()


In [ ]:

# ── D: CDS full-match per assembly ───────────────────────────────────────
if cds_asm.empty:
    print('Skipping D — no CDS data.')
else:
    pct_col = None
    for candidate in ['pct_Full_Match', 'pct_full_match', 'full_match_pct']:
        if candidate in cds_asm.columns:
            pct_col = candidate; break
    print(f'Using: {pct_col}')
    if pct_col:
        cds_vals = pd.to_numeric(cds_asm[pct_col], errors='coerce').dropna()
        cds_med = cds_vals.median()
        fig, ax = plt.subplots(figsize=(4.2, 5.0))
        ax.boxplot(cds_vals, vert=True, widths=0.5, patch_artist=True,
                   boxprops=dict(facecolor='#e76f51', alpha=0.3),
                   medianprops=dict(color='#264653', linewidth=2),
                   whiskerprops=dict(color='#264653'),
                   capprops=dict(color='#264653'),
                   flierprops=dict(marker='o', markersize=3, alpha=0.5))
        jitter = np.random.default_rng(42).uniform(-0.15, 0.15, len(cds_vals))
        ax.scatter(np.ones(len(cds_vals)) + jitter, cds_vals, s=6, alpha=0.35,
                   color='#e76f51', edgecolors='none', zorder=3)
        ax.set_ylabel('CDS Full Match (%)')
        ax.set_xticks([1]); ax.set_xticklabels([f'n = {len(cds_vals)} assemblies'])
        ax.text(1, cds_med + 0.15, f'{cds_med:.1f}%', ha='center', fontsize=9, fontweight='bold')
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        plt.tight_layout(); save_panel(fig, 'overview_D_cds_concordance'); plt.show()
    else:
        print('Could not find full-match column in CDS table.')


In [ ]:

# ── F4: CAT-only genes — biotype composition (median across assemblies) ──
if counts_df.empty:
    print('Skipping F4 — need counts_df.')
else:
    cat_genes = counts_df[counts_df['source']=='CAT'][['assembly_accession','gene_id','biotype_group']].drop_duplicates()
    ens_genes = counts_df[counts_df['source']=='Ensembl'][['assembly_accession','gene_id']].drop_duplicates()
    cat_only = (cat_genes.merge(ens_genes, on=['assembly_accession','gene_id'], how='left', indicator=True)
                         .query('_merge == "left_only"')
                         .drop(columns=['_merge']))

    if cat_only.empty:
        print('No CAT-only genes detected in counts tables.')
    else:
        per = (cat_only.groupby(['assembly_accession','biotype_group'])['gene_id']
                      .nunique().reset_index(name='n'))
        totals = per.groupby('assembly_accession')['n'].transform('sum')
        per['pct'] = 100 * per['n'] / totals
        med = (per.groupby('biotype_group')['pct'].median()
                  .reindex(BIOTYPE_ORDER, fill_value=0))

        fig, ax = plt.subplots(figsize=(6.6, 2.8))
        palette = {'protein_coding':'#2a9d8f','lncRNA':'#264653','pseudogene':'#457b9d','other_ncRNA':'#e9c46a','other':'#f4a261'}
        left = 0
        for b in BIOTYPE_ORDER:
            v = float(med.get(b, 0.0))
            ax.barh(0, v, left=left, height=0.55, color=palette.get(b,'#999'), edgecolor='white', linewidth=0.3, label=BIOTYPE_LABELS[b])
            if v >= 6:
                ax.text(left+v/2, 0, f'{v:.0f}%', ha='center', va='center', fontsize=7, color='white', fontweight='bold')
            left += v
        ax.set_xlim(0, 100); ax.set_yticks([]); ax.set_xlabel('CAT-only genes by biotype (median across assemblies)')
        ax.grid(axis='x', color='#ddd', linewidth=0.5, alpha=0.7); ax.set_axisbelow(True)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        handles = [mpatches.Patch(color=palette[b], label=BIOTYPE_LABELS[b]) for b in BIOTYPE_ORDER]
        fig = ax.get_figure()
        fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, -0.22), ncol=5, fontsize=7, frameon=False)
        plt.tight_layout(); save_panel(fig, 'overview_F4_cat_only_biotype'); plt.show()


In [ ]:

# ── F5: CAT-only genes — transcripts per gene (per-assembly medians) ─────
if counts_df.empty:
    print('Skipping F5 — need counts_df.')
else:
    cat = counts_df[counts_df['source']=='CAT'][['assembly_accession','gene_id','biotype_group','n_transcripts']].drop_duplicates()
    ens = counts_df[counts_df['source']=='Ensembl'][['assembly_accession','gene_id']].drop_duplicates()
    cat_only = (cat.merge(ens, on=['assembly_accession','gene_id'], how='left', indicator=True)
                   .query('_merge == "left_only"').drop(columns=['_merge']))

    if cat_only.empty:
        print('No CAT-only genes detected.')
    else:
        q = (cat_only.groupby(['assembly_accession','biotype_group'])['n_transcripts']
                    .median().reset_index(name='med_tx_per_gene'))
        fig, ax = plt.subplots(figsize=(7.8, 3.0))
        data = [q[q['biotype_group']==b]['med_tx_per_gene'].values for b in BIOTYPE_ORDER]
        parts = ax.violinplot(data, showmedians=True)
        for pc in parts['bodies']: pc.set_alpha(0.6)
        ax.set_xticks(range(1, len(BIOTYPE_ORDER)+1))
        ax.set_xticklabels([BIOTYPE_LABELS[b] for b in BIOTYPE_ORDER], rotation=20, ha='right')
        ax.set_ylabel('Median transcripts per CAT-only gene (per assembly)')
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        plt.tight_layout(); save_panel(fig, 'overview_F5_cat_only_tx_per_gene'); plt.show()
